# Oficina 09 — Validação cruzada na prática
### Aprendizado de Máquina Supervisionado — Curso de Inteligência Artificial
**FMU — 2026.2** · Prof. Luciano Tadeu Pereira · Aula 09 — 14/10/2026

---
Uma única divisão treino/teste depende de sorte. Aqui você vê, com experimentos curtos, por que a validação cruzada é mais honesta, e como um vazamento engana até quem sabe programar.

> **Como usar este notebook**
> 1. Rode as células **em ordem** (botão ▶ ou `Shift + Enter`).
> 2. Antes de cada código há um quadro **🎯 O que vamos fazer** e depois um quadro **🔎 Como ler o resultado**. Leia os dois: o código é o meio, a leitura é o objetivo.
> 3. No final há a seção **🧪 Hora de programar**: exercícios do mais fácil (★) ao mais desafiador (★★★). Cada um termina numa célula de conferência que mostra ✅ ou ❌.
> 4. Não precisa saber matemática avançada: onde aparece uma fórmula, há uma explicação em palavras logo ao lado.


## 1. Preparação

> 🎯 **O que vamos fazer:** carregar a base de crédito e separar treino e teste, como nas aulas anteriores.
>
> 💡 **Por que importa:** o teste fica guardado e só é aberto no final, como uma prova que o aluno não viu.

In [44]:
# Ferramenta de conferência dos exercícios (não precisa alterar)
def conferir(numero, teste, dica=""):
    """Roda o teste do exercício e diz se está certo. `teste` é uma função sem argumentos."""
    try:
        ok = teste()
    except Exception as erro:
        print(f"⏳ Exercício {numero}: ainda não está pronto ({type(erro).__name__}). Preencha o código acima e rode de novo.")
        return
    if ok:
        print(f"✅ Exercício {numero}: correto! Pode seguir.")
    else:
        print(f"❌ Exercício {numero}: ainda não. Dica: {dica}")

In [45]:
# Carrega a base preparada na aula 05 (base_credito_features.csv)
import os
ARQ = "base_credito_features.csv"
if not os.path.exists(ARQ):
    try:                                    # no Colab: abre a caixa para você enviar o arquivo do AVA
        from google.colab import files
        print("Arquivo não encontrado. Envie o base_credito_features.csv (está no AVA):")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Coloque {ARQ} na mesma pasta do notebook (o arquivo está no AVA).")

In [46]:
# Preparação: bibliotecas, divisão treino/teste e pré-processamento (igual aos laboratórios anteriores)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

d = pd.read_csv("base_credito_features.csv")

NUM = ["idade","renda_mensal","tempo_emprego_meses","valor_emprestimo","prazo_meses",
       "parcelas_atraso_12m","consultas_bureau_6m","score_bureau","parcela_estimada",
       "comprometimento_renda","emprestimo_sobre_renda"]
CAT = ["tipo_moradia","finalidade","uf"]

X, y = d[NUM + CAT], d["inadimplente"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore"))]), CAT),
])
cv = StratifiedKFold(5, shuffle=True, random_state=42)
print("treino:", X_tr.shape, "| teste:", X_te.shape, "| inadimplência:", round(y.mean()*100, 1), "%")

treino: (2250, 14) | teste: (750, 14) | inadimplência: 17.9 %


> 🔎 **Como ler o resultado:** a última linha mostra o tamanho dos dois blocos (2.250 clientes para treino, 750 para teste) e a taxa de inadimplência, cerca de 17,9%. Isso quer dizer que, em cada 100 clientes, uns 18 não pagam.

## 2. Cada dobra tem a mesma proporção de maus pagadores?

> 🎯 **O que vamos fazer:** comparar a divisão simples (`KFold`) com a divisão estratificada (`StratifiedKFold`) contando quantos inadimplentes caem em cada dobra.
>
> 💡 **Por que importa:** se uma dobra ficar sem inadimplentes, a nota daquela rodada não quer dizer nada. Estratificar é garantir que cada fatia tenha a mesma proporção, como dividir uma turma em grupos com o mesmo número de veteranos.

In [47]:
from sklearn.model_selection import KFold, StratifiedKFold

def positivos_por_dobra(divisor):
    return [int(y_tr.iloc[teste].sum()) for _, teste in divisor.split(X_tr, y_tr)]   # nº de inadimplentes em cada dobra de teste

simples = KFold(5, shuffle=True, random_state=0)
estrat  = StratifiedKFold(5, shuffle=True, random_state=0)

print("tamanho de cada dobra de teste:", len(X_tr)//5, "clientes")
print("inadimplentes por dobra, KFold simples :", positivos_por_dobra(simples))
print("inadimplentes por dobra, estratificado :", positivos_por_dobra(estrat))

tamanho de cada dobra de teste: 450 clientes
inadimplentes por dobra, KFold simples : [88, 77, 91, 82, 66]
inadimplentes por dobra, estratificado : [80, 81, 81, 81, 81]


> 🔎 **Como ler o resultado:** no estratificado, os cinco números são praticamente iguais (cerca de 80 inadimplentes por dobra, 17,9% de 450). No simples eles balançam um pouco mais. Com uma base pequena ou um evento mais raro (fraude, por exemplo), essa diferença vira problema sério.

## 3. A nota depende do sorteio?

> 🎯 **O que vamos fazer:** repetir a validação cruzada com cinco sorteios diferentes (cinco sementes) e ver o quanto a média muda.
>
> 💡 **Por que importa:** se mudar a semente muda a conclusão, a conclusão era sorte.

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

modelo = Pipeline([("prep", pre),
                   ("clf", LogisticRegression(C=0.1, l1_ratio=1.0, solver="liblinear",
                                              max_iter=3000, class_weight="balanced", random_state=42))])

medias = []
for semente in [0, 1, 2, 3, 4]:
    cv_s = StratifiedKFold(5, shuffle=True, random_state=semente)
    nota = cross_val_score(modelo, X_tr, y_tr, cv=cv_s, scoring="roc_auc")
    medias.append(nota.mean())
    print(f"semente {semente}: média = {nota.mean():.3f} | desvio entre as dobras = {nota.std():.3f}")

print(f"\nmédia das médias: {np.mean(medias):.3f} | variação entre sementes: {np.std(medias):.4f}")

semente 0: média = 0.721 | desvio entre as dobras = 0.020


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_

semente 1: média = 0.724 | desvio entre as dobras = 0.035


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_

semente 2: média = 0.719 | desvio entre as dobras = 0.021


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_

semente 3: média = 0.717 | desvio entre as dobras = 0.014
semente 4: média = 0.724 | desvio entre as dobras = 0.024

média das médias: 0.721 | variação entre sementes: 0.0026


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


> 🔎 **Como ler o resultado:** as médias mudam pouco entre sementes (na terceira casa), mas o **desvio entre dobras** (de 0,02 a 0,04) é maior. Moral: uma diferença de 0,01 entre dois modelos está dentro do ruído. Não declare vencedor por tão pouco.

## 4. O vazamento que engana: um experimento

> 🎯 **O que vamos fazer:** criar uma base de **ruído puro** (1.000 colunas de números sorteados para 100 clientes e um alvo sorteado, sem nenhuma relação entre eles) e ver o que acontece quando a escolha das melhores colunas é feita *fora* da validação.
>
> 💡 **Por que importa:** se não há nada para aprender, a nota deveria ser 0,5. Quando ela passa disso, o experimento está **vazando** informação do teste para o treino.

In [49]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(0)
X_ruido = rng.normal(size=(100, 1000))        # 100 clientes, 1.000 colunas de ruído
y_ruido = rng.integers(0, 2, 100)             # alvo sorteado: nada prevê nada

# ERRADO: escolher as 10 "melhores" colunas olhando o alvo de TODOS os clientes, e só depois validar
X_errado = SelectKBest(f_classif, k=10).fit_transform(X_ruido, y_ruido)
nota_errada = cross_val_score(LogisticRegression(), X_errado, y_ruido,
                              cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean()

# CERTO: a escolha das colunas fica dentro do Pipeline, refeita a cada dobra
certo = Pipeline([("sel", SelectKBest(f_classif, k=10)), ("clf", LogisticRegression())])
nota_certa = cross_val_score(certo, X_ruido, y_ruido,
                             cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean()

print(f"ROC-AUC fazendo a seleção FORA da validação : {nota_errada:.3f}   <- muito acima de 0,5 em dados SEM sinal")
print(f"ROC-AUC fazendo a seleção DENTRO do Pipeline: {nota_certa:.3f}   <- bem menor: sem o vazamento, a ilusão some")

ROC-AUC fazendo a seleção FORA da validação : 0.889   <- muito acima de 0,5 em dados SEM sinal
ROC-AUC fazendo a seleção DENTRO do Pipeline: 0.603   <- bem menor: sem o vazamento, a ilusão some


> 🔎 **Como ler o resultado:** os dados são ruído e mesmo assim o primeiro método 'prevê' bem. Isso é **vazamento**: ao escolher as colunas olhando todos os clientes, o teste já tinha espiado a resposta. Dentro do Pipeline a nota cai para perto de 0,6 (com só 100 clientes ainda sobra um pouco de sorte), muito mais perto da verdade, que é 0,5. Regra de bolso: se um passo 'aprende' com os dados (escolher colunas, padronizar, imputar), ele mora no Pipeline.

## 5. Interpretando sem inventar

> 🎯 **O que vamos fazer:** ler os coeficientes do modelo treinado e escrever o que eles autorizam dizer.
>
> 💡 **Por que importa:** coeficiente alto diz que o atributo pesa na **previsão**. Não diz que ele **causa** a inadimplência.

In [50]:
modelo.fit(X_tr, y_tr)
nomes = modelo.named_steps["prep"].get_feature_names_out()
coef = modelo.named_steps["clf"].coef_[0]
tab = (pd.DataFrame({"atributo": nomes, "coeficiente": coef.round(3), "razao_de_chances": np.exp(coef).round(3)})
         .assign(peso=lambda t: t.coeficiente.abs()).sort_values("peso", ascending=False).drop(columns="peso"))
display(tab.head(6))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


,atributo,coeficiente,razao_de_chances
7,num__score_bureau,-0.661,0.516
9,num__comprometimento_renda,0.429,1.535
12,cat__tipo_moradia_familiar,-0.302,0.740
15,cat__finalidade_capital_giro,-0.229,0.795
3,num__valor_emprestimo,-0.224,0.800
10,num__emprestimo_sobre_renda,0.213,1.238


> 🔎 **Como ler o resultado:** **razão de chances 1,63** (comprometimento da renda) quer dizer: para cada passo a mais nesse atributo, a chance de calote é cerca de 63% maior, mantido o resto. Isso descreve o modelo, não o mundo: não prova que reduzir o comprometimento reduziria o calote do cliente.

---
## 🧪 Hora de programar

### Exercício 1 ★ — Inadimplentes por dobra

Use `StratifiedKFold(5, shuffle=True, random_state=7)` e guarde em `por_dobra` a lista com o número de inadimplentes em cada dobra de teste (use a função `positivos_por_dobra`).

<details><summary>💡 Dica (clique só se travar)</summary>

A função já existe na seção 2: basta criar o divisor e passá-lo à função.

</details>

In [51]:
por_dobra = positivos_por_dobra (StratifiedKFold(5, shuffle=True, random_state=7))      # SUA VEZ
print(por_dobra)

[80, 81, 81, 81, 81]


In [52]:
conferir(1, lambda: por_dobra == positivos_por_dobra(StratifiedKFold(5, shuffle=True, random_state=7)) and max(por_dobra) - min(por_dobra) <= 1, 'chame positivos_por_dobra(StratifiedKFold(5, shuffle=True, random_state=7)).')

✅ Exercício 1: correto! Pode seguir.


### Exercício 2 ★★ — Validação repetida

Use `RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)` (15 notas ao todo) e guarde em `media_rep` e `desvio_rep` a média e o desvio.

<details><summary>💡 Dica (clique só se travar)</summary>

`from sklearn.model_selection import RepeatedStratifiedKFold`. Passe-o como `cv=` em `cross_val_score`.

</details>

In [53]:
from sklearn.model_selection import RepeatedStratifiedKFold
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3,  random_state=42)
notas = cross_val_score(modelo, X_tr, y_tr,  cv=cv_rep, scoring='roc_auc')
media_rep = notas.mean()       # SUA VEZ
desvio_rep = notas.std()
print(media_rep, desvio_rep)


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_

0.720555923659281 0.03286504908719924


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


In [54]:
conferir(2, lambda: (lambda n: abs(media_rep - n.mean()) < 1e-9 and abs(desvio_rep - n.std()) < 1e-9)(cross_val_score(modelo, X_tr, y_tr, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42), scoring='roc_auc')), "cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42) em cross_val_score(modelo, X_tr, y_tr, scoring='roc_auc').")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_

✅ Exercício 2: correto! Pode seguir.


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


### Exercício 3 ★★ — Conserte o vazamento

O código abaixo padroniza **antes** de validar. Reescreva usando um `Pipeline` com `StandardScaler` e `LogisticRegression` e guarde a média em `nota_certa_ex`. Use só as colunas numéricas (`X_tr[NUM]`, com `fillna(0)` para simplificar) e `cv`.

<details><summary>💡 Dica (clique só se travar)</summary>

`Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=3000))])` e então `cross_val_score(pipe, X_tr[NUM].fillna(0), y_tr, cv=cv, scoring='roc_auc')`.

</details>

In [55]:
# versão ERRADA (apenas para ler): padroniza todos os dados antes de dividir em dobras
# X_p = StandardScaler().fit_transform(X_tr[NUM].fillna(0)); cross_val_score(LogisticRegression(max_iter=3000), X_p, y_tr, cv=cv)

nota_certa_ex = cross_val_score(
    Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=3000))]),
    X_tr[NUM].fillna(0), y_tr, cv=cv, scoring='roc_auc'
).mean()    # SUA VEZ: versão certa, com Pipeline
print(nota_certa_ex)

0.6902832511825284


In [56]:
conferir(3, lambda: abs(nota_certa_ex - cross_val_score(Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=3000))]), X_tr[NUM].fillna(0), y_tr, cv=cv, scoring='roc_auc').mean()) < 1e-9, 'o StandardScaler precisa estar dentro do Pipeline, junto com a LogisticRegression, e cross_val_score recebe o pipe.')

✅ Exercício 3: correto! Pode seguir.


### Exercício 4 ★★★ — A frase que a tabela autoriza

Usando a tabela `tab`, guarde em `or_comprometimento` a razão de chances de `num__comprometimento_renda` e em `afirmacao_correta` a letra da **única** afirmação que a tabela autoriza:

- **A)** Reduzir o comprometimento do cliente reduz o calote dele.
- **B)** No modelo, mais comprometimento está associado a maior chance prevista de calote.
- **C)** O comprometimento causa a inadimplência.

<details><summary>💡 Dica (clique só se travar)</summary>

Procure a linha com `tab.atributo == 'num__comprometimento_renda'`. Lembre: coeficiente descreve o modelo, não prova causa.

</details>

In [57]:
or_comprometimento = float(tab.loc[tab.atributo == 'num__comprometimento_renda', 'razao_de_chances'].iloc[0])   # SUA VEZ
afirmacao_correta = 'B'    # 'A', 'B' ou 'C'
print(or_comprometimento, afirmacao_correta)

1.535 B


In [58]:
conferir(4, lambda: abs(or_comprometimento - float(tab.loc[tab.atributo == 'num__comprometimento_renda', 'razao_de_chances'].iloc[0])) < 1e-9 and afirmacao_correta == 'B', 'a única afirmação que não diz causa nem promete efeito é a B (descreve o modelo).')

✅ Exercício 4: correto! Pode seguir.


---
## 📤 Entrega

Salve uma cópia e entregue o link no AVA com os quatro ✅. O experimento da seção 4 costuma virar questão de prova: saiba explicá-lo com as suas palavras.